# Module 09: Inspecting fMRIPrep Outputs

fMRIPrep produces a rich set of derivatives — preprocessed BOLD images, brain masks, confound tables, and HTML quality reports. Before running any first-level GLM, you need to understand what is in these outputs and which confound columns to include.

This notebook walks through:

1. Exploring the fMRIPrep derivatives directory structure
2. Reading and understanding the confounds timeseries TSV
3. Plotting the six rigid-body motion parameters
4. Calculating and visualising framewise displacement (FD)
5. Identifying and censoring high-motion volumes (scrubbing)
6. Selecting a confound regression strategy for GLM
7. Loading and visualising the preprocessed BOLD image and brain mask
8. Navigating the fMRIPrep HTML quality report

---

**Learning objectives:** After completing this notebook you will be able to:
1. Navigate the fMRIPrep derivatives directory and locate key output files
2. Load and inspect the confounds TSV with pandas
3. Plot motion parameters and FD, and identify motion spikes
4. Create a scrubbing mask to exclude high-motion volumes
5. Select minimal, moderate, or aggressive confound columns for GLM
6. Visualise the preprocessed BOLD and brain mask with nilearn
7. Interpret sections of the fMRIPrep HTML report

## 1  fMRIPrep Derivatives Structure

fMRIPrep writes outputs to a `derivatives/fmriprep/` directory following the BIDS Derivatives specification. Each subject gets its own folder containing an `anat/` subdirectory for structural outputs, a `func/` subdirectory for functional outputs, and a `figures/` subdirectory for QC images.

The cell below walks the derivatives tree for a given subject and lists the files found.

In [ ]:
import os
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Configuration — edit these paths to point to your fMRIPrep derivatives.
# ---------------------------------------------------------------------------
FMRIPREP_DIR = Path("../../derivatives/fmriprep")  # adjust as needed
SUBJECT = "01"
TASK = "rest"

sub_dir = FMRIPREP_DIR / f"sub-{SUBJECT}"

if sub_dir.exists():
    print(f"Found derivatives for sub-{SUBJECT}:\n")
    for root, dirs, files in os.walk(sub_dir):
        # Skip hidden directories
        dirs[:] = [d for d in sorted(dirs) if not d.startswith(".")]
        level = Path(root).relative_to(FMRIPREP_DIR).parts
        indent = "    " * (len(level) - 1)
        print(f"{indent}{Path(root).name}/")
        file_indent = "    " * len(level)
        for fname in sorted(files):
            print(f"{file_indent}{fname}")
else:
    print("fMRIPrep derivatives not found at expected path.")
    print("Showing a representative synthetic directory structure instead:\n")
    synthetic_tree = [
        "sub-01/",
        "    anat/",
        "        sub-01_desc-brain_mask.nii.gz",
        "        sub-01_desc-preproc_T1w.nii.gz",
        "        sub-01_from-T1w_to-MNI152NLin2009cAsym_mode-image_xfm.h5",
        "        sub-01_from-MNI152NLin2009cAsym_to-T1w_mode-image_xfm.h5",
        "    func/",
        "        sub-01_task-rest_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz",
        "        sub-01_task-rest_space-MNI152NLin2009cAsym_desc-brain_mask.nii.gz",
        "        sub-01_task-rest_desc-confounds_timeseries.tsv",
        "        sub-01_task-rest_desc-confounds_timeseries.json",
        "        sub-01_task-rest_space-MNI152NLin2009cAsym_boldref.nii.gz",
        "    figures/",
        "        sub-01_task-rest_desc-carpetplot_bold.svg",
        "        sub-01_task-rest_desc-rois_bold.svg",
        "        sub-01_desc-reconall_T1w.svg",
    ]
    for line in synthetic_tree:
        print(line)

### 1.1  Locating key functional output files

The three most important outputs in `func/` are:

| File | Description |
|------|-------------|
| `*_desc-preproc_bold.nii.gz` | Preprocessed BOLD (motion-corrected, distortion-corrected, registered to output space) |
| `*_desc-brain_mask.nii.gz` | Binary brain mask in the same space |
| `*_desc-confounds_timeseries.tsv` | One row per volume, one column per confound variable |

In [ ]:
import glob

func_dir = sub_dir / "func"

# Patterns for the three key files
patterns = {
    "Preprocessed BOLD": f"sub-{SUBJECT}_task-{TASK}*_desc-preproc_bold.nii.gz",
    "Brain mask": f"sub-{SUBJECT}_task-{TASK}*_desc-brain_mask.nii.gz",
    "Confounds TSV": f"sub-{SUBJECT}_task-{TASK}*_desc-confounds_timeseries.tsv",
}

found_files = {}
for label, pattern in patterns.items():
    matches = sorted(func_dir.glob(pattern)) if func_dir.exists() else []
    if matches:
        found_files[label] = matches[0]
        print(f"{label}:\n  {matches[0]}")
    else:
        print(f"{label}: NOT FOUND (will use synthetic data below)")

BOLD_FILE = found_files.get("Preprocessed BOLD")
MASK_FILE = found_files.get("Brain mask")
CONFOUNDS_FILE = found_files.get("Confounds TSV")

## 2  Reading the Confounds TSV

The confounds timeseries file is a tab-separated table where each **row** corresponds to one fMRI volume and each **column** is a confound variable estimated by fMRIPrep. Common columns include:

- **Motion parameters:** `trans_x`, `trans_y`, `trans_z`, `rot_x`, `rot_y`, `rot_z`
- **Motion derivatives:** `trans_x_derivative1`, etc.
- **Motion summary:** `framewise_displacement`, `rmsd`
- **Tissue signals:** `global_signal`, `white_matter`, `csf`
- **CompCor:** `a_comp_cor_00`–`a_comp_cor_XX` (anatomical), `t_comp_cor_00`–`t_comp_cor_XX` (temporal)
- **ICA-AROMA** (if enabled): `aroma_motion_XX`, `aroma_noise_XX`
- **Cosine drift terms:** `cosine00`, `cosine01`, etc.

The first row may contain `n/a` for columns that require a previous timepoint (e.g., derivatives and FD).

In [ ]:
import numpy as np
import pandas as pd

# Add repo root to sys.path so utils/ is importable
repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from utils.io_utils import load_tsv
    USE_UTILS = True
except ImportError:
    USE_UTILS = False
    print("utils.io_utils not found; falling back to pandas.read_csv")

N_VOLUMES = 200  # length of synthetic timeseries
rng = np.random.default_rng(seed=42)

if CONFOUNDS_FILE is not None:
    if USE_UTILS:
        confounds_df = load_tsv(str(CONFOUNDS_FILE))
    else:
        confounds_df = pd.read_csv(CONFOUNDS_FILE, sep="\t")
    N_VOLUMES = len(confounds_df)
    print(f"Loaded confounds: {confounds_df.shape[0]} volumes × {confounds_df.shape[1]} columns")
    print("\nFirst 5 columns:")
    display(confounds_df.iloc[:5, :5])
else:
    # Generate realistic synthetic confounds
    time = np.arange(N_VOLUMES)

    # Slow drift + small random motion
    def _motion_param(amp=0.3, drift_amp=0.1):
        drift = drift_amp * np.sin(2 * np.pi * time / N_VOLUMES)
        noise = rng.normal(0, amp * 0.2, N_VOLUMES)
        ts = np.cumsum(noise) * 0.05 + drift
        ts -= ts.mean()
        return ts

    trans_x = _motion_param(0.4)
    trans_y = _motion_param(0.3)
    trans_z = _motion_param(0.5)
    rot_x   = _motion_param(0.005)
    rot_y   = _motion_param(0.004)
    rot_z   = _motion_param(0.006)

    # Inject a few motion spikes
    spike_vols = [40, 41, 95, 150]
    for sv in spike_vols:
        trans_x[sv] += rng.choice([-1, 1]) * rng.uniform(1.0, 2.5)
        trans_y[sv] += rng.choice([-1, 1]) * rng.uniform(0.5, 1.5)
        rot_z[sv]   += rng.choice([-1, 1]) * rng.uniform(0.02, 0.05)

    def _derivative(x):
        d = np.diff(x, prepend=x[0])
        d[0] = np.nan
        return d

    global_signal = rng.normal(0, 1, N_VOLUMES)
    white_matter  = rng.normal(0, 0.5, N_VOLUMES)
    csf           = rng.normal(0, 0.5, N_VOLUMES)

    # Anatomical CompCor components (orthogonal random signals)
    acompcor = {f"a_comp_cor_{i:02d}": rng.normal(0, 1, N_VOLUMES) for i in range(6)}

    confounds_df = pd.DataFrame({
        "trans_x": trans_x,
        "trans_y": trans_y,
        "trans_z": trans_z,
        "rot_x":   rot_x,
        "rot_y":   rot_y,
        "rot_z":   rot_z,
        "trans_x_derivative1": _derivative(trans_x),
        "trans_y_derivative1": _derivative(trans_y),
        "trans_z_derivative1": _derivative(trans_z),
        "rot_x_derivative1":   _derivative(rot_x),
        "rot_y_derivative1":   _derivative(rot_y),
        "rot_z_derivative1":   _derivative(rot_z),
        "global_signal": global_signal,
        "white_matter":  white_matter,
        "csf":           csf,
        **acompcor,
    })

    # Compute framewise displacement from motion parameters
    rot_mm = 50.0  # approximate head radius in mm
    d_trans = confounds_df[["trans_x", "trans_y", "trans_z"]].diff().abs()
    d_rot   = confounds_df[["rot_x", "rot_y", "rot_z"]].diff().abs() * rot_mm
    fd = d_trans.sum(axis=1) + d_rot.sum(axis=1)
    fd.iloc[0] = np.nan
    confounds_df.insert(6, "framewise_displacement", fd)

    print("[Synthetic data] Generated confounds timeseries.")
    print(f"Shape: {confounds_df.shape[0]} volumes × {confounds_df.shape[1]} columns")
    print("\nFirst 5 rows (selected columns):")
    display(confounds_df[["trans_x", "trans_y", "trans_z", "framewise_displacement"]].head())

In [ ]:
# Show all available column names grouped by type
all_cols = confounds_df.columns.tolist()

groups = {
    "Motion (6-param)": [c for c in all_cols if c in ["trans_x","trans_y","trans_z","rot_x","rot_y","rot_z"]],
    "Motion derivatives": [c for c in all_cols if "derivative1" in c and "power2" not in c],
    "Motion squares": [c for c in all_cols if "power2" in c],
    "Motion summary": [c for c in all_cols if c in ["framewise_displacement", "rmsd"]],
    "Tissue signals": [c for c in all_cols if c in ["global_signal", "white_matter", "csf"]],
    "aCompCor": [c for c in all_cols if c.startswith("a_comp_cor")],
    "tCompCor": [c for c in all_cols if c.startswith("t_comp_cor")],
    "ICA-AROMA": [c for c in all_cols if "aroma" in c.lower()],
    "Cosine drift": [c for c in all_cols if c.startswith("cosine")],
    "Other": [],
}

labelled = {c for cols in groups.values() for c in cols}
groups["Other"] = [c for c in all_cols if c not in labelled]

for group, cols in groups.items():
    if cols:
        print(f"{group} ({len(cols)}): {', '.join(cols)}")

## 3  Plotting Motion Parameters

The six rigid-body motion parameters describe how the subject's head moved relative to a reference volume:
- **Translations** (`trans_x/y/z`): displacement in mm along each axis
- **Rotations** (`rot_x/y/z`): rotation in radians around each axis

Inspecting these plots lets you identify:
- **Slow drift** — gradual motion that may indicate discomfort or fatigue
- **Step changes** — abrupt head movement (often seen as vertical jumps)
- **Excessive range** — rule-of-thumb: > 1 voxel translation or > 1° rotation is concerning

The `utils.plotting.plot_motion_params` helper produces a two-panel figure (translations on top, rotations below).

In [ ]:
import matplotlib.pyplot as plt
import tempfile

# Try to use utils.plotting; fall back to manual plotting
try:
    from utils.plotting import plot_motion_params

    if CONFOUNDS_FILE is not None:
        fig = plot_motion_params(str(CONFOUNDS_FILE))
    else:
        # Write synthetic confounds to a temp file so the helper can load it
        with tempfile.NamedTemporaryFile(suffix=".tsv", delete=False, mode="w") as tmp:
            tmp_path = tmp.name
        confounds_df.to_csv(tmp_path, sep="\t", index=False)
        fig = plot_motion_params(tmp_path)
        os.unlink(tmp_path)
    plt.suptitle("Motion Parameters", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

except Exception as exc:
    print(f"utils.plotting.plot_motion_params unavailable ({exc}); using manual plot.")

    motion_cols_trans = ["trans_x", "trans_y", "trans_z"]
    motion_cols_rot   = ["rot_x",   "rot_y",   "rot_z"]
    colors_trans = ["#e74c3c", "#27ae60", "#2980b9"]
    colors_rot   = ["#e67e22", "#8e44ad", "#16a085"]

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    for col, color in zip(motion_cols_trans, colors_trans):
        axes[0].plot(confounds_df[col], label=col, color=color, linewidth=0.9)
    axes[0].axhline(0, color="black", linewidth=0.5, linestyle="--")
    axes[0].set_ylabel("Translation (mm)")
    axes[0].legend(loc="upper right", fontsize=8)
    axes[0].set_title("Translations")

    for col, color in zip(motion_cols_rot, colors_rot):
        axes[1].plot(np.degrees(confounds_df[col]), label=col, color=color, linewidth=0.9)
    axes[1].axhline(0, color="black", linewidth=0.5, linestyle="--")
    axes[1].set_ylabel("Rotation (degrees)")
    axes[1].set_xlabel("Volume")
    axes[1].legend(loc="upper right", fontsize=8)
    axes[1].set_title("Rotations")

    plt.suptitle("Motion Parameters", fontsize=14)
    plt.tight_layout()
    plt.show()

## 4  Framewise Displacement (FD)

Framewise displacement (Power et al., 2012) summarises head movement between consecutive volumes as a single scalar per timepoint:

$$\text{FD}_t = |\Delta d_x| + |\Delta d_y| + |\Delta d_z| + r|\Delta \theta_x| + r|\Delta \theta_y| + r|\Delta \theta_z|$$

where $\Delta$ denotes the difference between volume $t$ and $t-1$, and $r = 50\,\text{mm}$ converts rotations to approximate displacement on a sphere of radius $r$ (a standard approximation for head radius).

**Common thresholds:**
- `FD > 0.5 mm` — widely used for scrubbing (Power et al., 2012)
- `FD > 0.2 mm` — more conservative (recommended for high-resolution or paediatric data)

The mean FD across a run is a useful summary statistic for comparing motion levels across subjects.

In [ ]:
FD_THRESHOLD = 0.5  # mm

fd_series = confounds_df["framewise_displacement"].copy()

# Replace n/a strings with NaN if they came from a real TSV file
fd_series = pd.to_numeric(fd_series, errors="coerce")

fd_valid = fd_series.dropna()
mean_fd = fd_valid.mean()
max_fd  = fd_valid.max()
n_spikes = (fd_valid > FD_THRESHOLD).sum()
pct_spikes = 100 * n_spikes / len(fd_valid)

print(f"Mean FD          : {mean_fd:.4f} mm")
print(f"Max FD           : {max_fd:.4f} mm")
print(f"Volumes > {FD_THRESHOLD} mm FD: {n_spikes} ({pct_spikes:.1f}%)")

# Plot FD over time
fig, ax = plt.subplots(figsize=(12, 3))
volumes = np.arange(len(fd_series))

ax.plot(volumes, fd_series, color="steelblue", linewidth=0.9, label="Framewise displacement")
ax.axhline(FD_THRESHOLD, color="red", linewidth=1.2, linestyle="--",
           label=f"Threshold ({FD_THRESHOLD} mm)")

# Shade volumes that exceed the threshold
spike_mask = fd_series > FD_THRESHOLD
ax.fill_between(volumes, 0, fd_series.fillna(0),
                where=spike_mask, color="red", alpha=0.3, label="Motion spike")

ax.set_xlabel("Volume")
ax.set_ylabel("FD (mm)")
ax.set_title(f"Framewise Displacement  |  mean={mean_fd:.3f} mm, {n_spikes} spikes > {FD_THRESHOLD} mm")
ax.legend(loc="upper right", fontsize=8)
ax.set_xlim(0, len(fd_series) - 1)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## 5  Scrubbing and Censoring High-Motion Volumes

**Scrubbing** (Power et al., 2012) removes high-motion volumes from analysis to avoid motion artefacts corrupting connectivity estimates or task-evoked responses. Two common approaches:

1. **Complete removal** — delete the flagged volumes from the timeseries entirely (reduces degrees of freedom but removes contaminated data)
2. **Spike regression** — add a binary indicator regressor (one column per spike) to the GLM design matrix, which effectively conditions out those volumes without removing them

It is also common to flag the volumes immediately **before and after** a spike because the HRF and autocorrelation can spread motion artefact to adjacent timepoints.

> **Practical tip:** If more than 20–25% of volumes are flagged you should consider excluding the entire run.

In [ ]:
def create_scrubbing_mask(
    fd_series: pd.Series,
    threshold: float = 0.5,
    extend_before: int = 0,
    extend_after: int = 0,
) -> np.ndarray:
    """Create a boolean scrubbing mask; True = volume to KEEP.

    Args:
        fd_series: Framewise displacement values (NaN allowed for first volume).
        threshold: FD threshold above which volumes are censored.
        extend_before: Number of volumes before each spike to also censor.
        extend_after: Number of volumes after each spike to also censor.

    Returns:
        Boolean array, same length as fd_series; True = keep.
    """
    fd_vals = pd.to_numeric(fd_series, errors="coerce").to_numpy()
    n = len(fd_vals)
    censor = np.zeros(n, dtype=bool)  # True = censor

    for i, val in enumerate(fd_vals):
        if np.isnan(val) or val > threshold:
            for j in range(max(0, i - extend_before), min(n, i + extend_after + 1)):
                censor[j] = True

    return ~censor  # return keep mask


# Standard scrubbing: FD > 0.5 mm, plus one volume after each spike
keep_mask = create_scrubbing_mask(fd_series, threshold=FD_THRESHOLD, extend_after=1)

n_keep   = keep_mask.sum()
n_censor = (~keep_mask).sum()

print(f"Total volumes  : {len(keep_mask)}")
print(f"Kept           : {n_keep} ({100*n_keep/len(keep_mask):.1f}%)")
print(f"Censored       : {n_censor} ({100*n_censor/len(keep_mask):.1f}%)")

# Visualise the scrubbing mask
fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1]})

axes[0].plot(fd_series.values, color="steelblue", linewidth=0.9)
axes[0].axhline(FD_THRESHOLD, color="red", linestyle="--", linewidth=1.0,
                label=f"FD threshold ({FD_THRESHOLD} mm)")
axes[0].set_ylabel("FD (mm)")
axes[0].legend(loc="upper right", fontsize=8)
axes[0].set_title("Framewise Displacement with Scrubbing Mask")

# Show keep (white) vs censor (red) per volume
mask_img = (~keep_mask).astype(float).reshape(1, -1)
axes[1].imshow(mask_img, aspect="auto", cmap="RdYlGn_r",
               vmin=0, vmax=1, extent=[0, len(keep_mask), 0, 1])
axes[1].set_yticks([])
axes[1].set_xlabel("Volume")
axes[1].set_ylabel("Censor")

plt.tight_layout()
plt.show()

# Create a spike-regression binary column (useful for GLM)
motion_outlier = (~keep_mask).astype(int)
print(f"\nScrubbing column 'motion_outlier': {motion_outlier.sum()} censored volumes flagged")

## 6  Selecting Confound Columns for GLM

Ciric et al. (2017) systematically evaluated many confound regression strategies for resting-state fMRI. Three commonly used strategies are summarised below:

| Strategy | Columns | Notes |
|----------|---------|-------|
| **Minimal** | 6 motion params + FD | Fewest degrees of freedom consumed; may leave physiological noise |
| **Moderate** | 6 motion + global/WM/CSF signals + 6 aCompCor | Good balance; recommended default for task fMRI |
| **Aggressive** | 24–36 motion params + extended aCompCor | Best artefact removal; recommended for resting-state; expensive in df |

The cell below extracts a confound matrix for each strategy and compares the number of regressors.

In [ ]:
def select_confounds(
    df: pd.DataFrame,
    strategy: str = "moderate",
    fd_threshold: float = 0.5,
    add_scrubbing: bool = False,
) -> pd.DataFrame:
    """Extract a confound matrix using the requested denoising strategy.

    Args:
        df: Full confounds DataFrame as loaded from fMRIPrep.
        strategy: One of 'minimal', 'moderate', or 'aggressive'.
        fd_threshold: FD threshold for scrubbing (used when add_scrubbing=True).
        add_scrubbing: If True, append a binary 'motion_outlier' column.

    Returns:
        DataFrame containing only the selected confound columns.
    """
    available = set(df.columns)

    MOTION_6 = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]
    MOTION_DERIV = [f"{p}_derivative1" for p in MOTION_6]
    MOTION_SQ    = [f"{p}_power2" for p in MOTION_6]
    MOTION_DERIV_SQ = [f"{p}_derivative1_power2" for p in MOTION_6]
    TISSUE = ["global_signal", "white_matter", "csf"]
    ACOMPCOR_6 = [f"a_comp_cor_{i:02d}" for i in range(6)]
    ACOMPCOR_ALL = sorted([c for c in available if c.startswith("a_comp_cor")])

    if strategy == "minimal":
        desired = MOTION_6 + ["framewise_displacement"]
    elif strategy == "moderate":
        desired = MOTION_6 + ["framewise_displacement"] + TISSUE + ACOMPCOR_6
    elif strategy == "aggressive":
        desired = (MOTION_6 + MOTION_DERIV + MOTION_SQ + MOTION_DERIV_SQ
                   + ["framewise_displacement"] + TISSUE + ACOMPCOR_ALL)
    else:
        raise ValueError(f"Unknown strategy '{strategy}'. Choose: minimal, moderate, aggressive")

    # Keep only columns that actually exist, warn about missing ones
    selected, missing = [], []
    for col in desired:
        if col in available:
            selected.append(col)
        else:
            missing.append(col)

    if missing:
        print(f"  [Warning] {len(missing)} column(s) not found and skipped: {missing[:5]}{'...' if len(missing)>5 else ''}")

    result = df[selected].copy()

    if add_scrubbing and "framewise_displacement" in available:
        fd_vals = pd.to_numeric(df["framewise_displacement"], errors="coerce")
        result["motion_outlier"] = (fd_vals > fd_threshold).astype(int)

    return result


strategies = ["minimal", "moderate", "aggressive"]
for strategy in strategies:
    cm = select_confounds(confounds_df, strategy=strategy, add_scrubbing=True)
    print(f"\n{'='*50}")
    print(f"Strategy: {strategy.upper()}")
    print(f"  Regressors : {cm.shape[1]}")
    print(f"  Columns    : {', '.join(cm.columns.tolist())}")

In [ ]:
# Visualise the moderate confound matrix as a heatmap
moderate_cm = select_confounds(confounds_df, strategy="moderate")

# Standardise each column for display
cm_norm = (moderate_cm - moderate_cm.mean()) / (moderate_cm.std() + 1e-8)
cm_norm = cm_norm.fillna(0)  # first-row NaN from FD becomes 0

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(cm_norm.T.values, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
ax.set_yticks(range(len(cm_norm.columns)))
ax.set_yticklabels(cm_norm.columns, fontsize=7)
ax.set_xlabel("Volume")
ax.set_title("Moderate Confound Matrix (z-scored columns)")
fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label="z-score")
plt.tight_layout()
plt.show()

## 7  Inspecting the Brain Mask and BOLD Data

After confirming the confounds look reasonable, it is worth visually inspecting the preprocessed BOLD image and its brain mask.

- The **brain mask** should cover the whole brain without including too much extra-cerebral signal or missing any cortical regions.
- The **mean BOLD image** (average across volumes) should look anatomically plausible — check for signal dropout in frontal and temporal regions (common near air-tissue interfaces).

We use `nibabel` to load the images and `nilearn.plotting` to create informative glass-brain and mosaic views.

In [ ]:
try:
    import nibabel as nib
    from nilearn import image as nli
    from nilearn import plotting as nlp

    if BOLD_FILE is not None and MASK_FILE is not None:
        bold_img = nib.load(str(BOLD_FILE))
        mask_img = nib.load(str(MASK_FILE))

        print(f"BOLD shape  : {bold_img.shape}  (x, y, z, time)")
        print(f"Voxel size  : {bold_img.header.get_zooms()[:3]} mm")
        print(f"TR          : {bold_img.header.get_zooms()[3]:.3f} s")
        print(f"Mask shape  : {mask_img.shape}")

        # Compute mean BOLD image
        mean_bold = nli.mean_img(bold_img)

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Mean BOLD
        display1 = nlp.plot_anat(
            mean_bold, title="Mean BOLD",
            display_mode="mosaic", axes=axes[0],
            cut_coords=5,
        )

        # Brain mask overlaid on mean BOLD
        display2 = nlp.plot_roi(
            mask_img, bg_img=mean_bold,
            title="Brain Mask (red) on Mean BOLD",
            display_mode="mosaic", axes=axes[1],
            cut_coords=5, alpha=0.4, colorbar=False,
        )

        plt.tight_layout()
        plt.show()

    else:
        print("Real BOLD/mask files not found. Generating synthetic NIfTI for demonstration.")

        # Create a tiny synthetic 4-D image for illustration
        shape_3d = (30, 36, 30)
        n_vols   = 20
        affine   = np.diag([3, 3, 3, 1])  # 3mm isotropic

        # Spherical brain region
        grid = np.mgrid[:shape_3d[0], :shape_3d[1], :shape_3d[2]]
        centre = np.array(shape_3d) / 2.0
        dist   = np.sqrt(sum((grid[i] - centre[i]) ** 2 for i in range(3)))
        mask_data = (dist < 12).astype(np.float32)

        bold_data = (mask_data[..., np.newaxis]
                     * (800 + 50 * rng.standard_normal(shape_3d + (n_vols,))))
        bold_data = bold_data.astype(np.float32)

        synth_bold = nib.Nifti1Image(bold_data, affine)
        synth_mask = nib.Nifti1Image(mask_data, affine)

        mean_bold = nli.mean_img(synth_bold)

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        nlp.plot_anat(mean_bold, title="[Synthetic] Mean BOLD",
                      display_mode="z", cut_coords=5, axes=axes[0])
        nlp.plot_roi(synth_mask, bg_img=mean_bold,
                     title="[Synthetic] Brain Mask on Mean BOLD",
                     display_mode="z", cut_coords=5, axes=axes[1], alpha=0.5)
        plt.tight_layout()
        plt.show()

except ImportError as exc:
    print(f"nibabel/nilearn not available ({exc}).")
    print("Install with: pip install nibabel nilearn")

In [ ]:
# BOLD signal quality: plot global mean signal over time and check for spikes
try:
    import nibabel as nib
    from nilearn import image as nli
    from nilearn.maskers import NiftiMasker

    if BOLD_FILE is not None and MASK_FILE is not None:
        masker = NiftiMasker(mask_img=str(MASK_FILE), standardize=False)
        bold_masked = masker.fit_transform(str(BOLD_FILE))  # (n_vols, n_voxels)
        global_mean = bold_masked.mean(axis=1)
    else:
        global_mean = confounds_df["global_signal"].to_numpy() if "global_signal" in confounds_df else None

    if global_mean is not None:
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(global_mean, color="#2c3e50", linewidth=0.9)
        ax.set_xlabel("Volume")
        ax.set_ylabel("Mean signal (a.u.)")
        ax.set_title("Global Mean BOLD Signal Over Time")
        plt.tight_layout()
        plt.show()

except Exception as exc:
    print(f"Could not compute global mean signal: {exc}")

## 8  The fMRIPrep HTML Report

fMRIPrep generates a self-contained HTML report for each subject at `derivatives/fmriprep/sub-XX.html`. Open it in any web browser. The report is organised into sections:

### Anatomical section
- **Brain mask and brain tissue segmentation:** Overlays of the estimated brain mask and grey-matter / white-matter / CSF tissue probability maps on the T1w image. Check that the mask is tight and the tissue boundaries look correct.
- **Spatial normalisation:** Shows the T1w registered to MNI space. Verify that major landmarks (corpus callosum, cerebellum, cortex) are well-aligned.
- **Surface reconstruction** (if FreeSurfer ran): Pial and white-matter surfaces overlaid on the T1w. Look for topology errors or missing surface patches.

### Functional section (one per BOLD run)
- **Summary statistics:** TR, number of volumes, output space, task name.
- **Susceptibility distortion correction:** Before/after comparison of EPI distortions if a fieldmap was used.
- **Alignment:** EPI registered to T1w — check that cortical boundaries align across the two images.
- **Brain mask:** Functional brain mask overlaid on the EPI mean. Check for dropout or over-inclusion.
- **ROI plot:** Outlines of white-matter and CSF ROIs used for CompCor, plotted on the mean EPI.
- **Carpet plot:** A summary of the BOLD timeseries for all voxels (grey matter, white matter, CSF) plotted as a heat map over time. The top panel shows FD and DVARS. Carpet plots are very useful for identifying:
  - Motion spikes (vertical striped artefacts)
  - Respiration-related signals
  - Global signal fluctuations (horizontal bands)
  - Ghosting or signal instability at the start of the run

### How to use the report for QC

1. Check the **mean FD and DVARS** values printed in the functional summary.
2. Inspect the **carpet plot** — a clean run should show no large vertical bands after the initial stabilisation period.
3. Verify the **EPI-to-T1w alignment** by toggling between the two images.
4. Check that the **brain mask** does not cut off prefrontal or temporal regions.
5. If FreeSurfer surfaces look implausible, this may affect CompCor ROI placement.

In [ ]:
# Check if the HTML report exists and show its size
html_report = FMRIPREP_DIR / f"sub-{SUBJECT}.html"

if html_report.exists():
    size_mb = html_report.stat().st_size / 1e6
    print(f"HTML report found: {html_report}")
    print(f"File size: {size_mb:.1f} MB")
    print("\nOpen this file in your browser to inspect the full QC report.")
else:
    print(f"HTML report not found at: {html_report}")
    print("After running fMRIPrep, the report will be at:")
    print(f"  derivatives/fmriprep/sub-{SUBJECT}.html")

## 9  Summary

In this module you have:

- **Explored** the fMRIPrep derivatives structure and identified the key output files for each subject
- **Loaded** the confounds timeseries TSV and examined its columns
- **Plotted** the six rigid-body motion parameters and visually identified step changes and drift
- **Calculated** framewise displacement (FD) and identified motion spikes above the 0.5 mm threshold
- **Created** a scrubbing mask to censor high-motion volumes and learned two censoring strategies
- **Extracted** confound matrices for minimal, moderate, and aggressive denoising strategies
- **Visualised** the preprocessed BOLD image and brain mask using nilearn
- **Learned** to interpret the fMRIPrep HTML quality report, especially the carpet plot

### Next Steps

You are now ready to proceed to first-level GLM modelling. Before doing so, run `scripts/inspect_fmriprep_outputs.py` on all your subjects to generate a motion summary table. Use `scripts/extract_confounds.py` to save the confound matrix for each run with your chosen strategy. Flag and consider excluding subjects or runs where:

- Mean FD > 0.5 mm
- More than 20% of volumes are censored
- The carpet plot shows clear structured artefacts

### References

- Esteban O, et al. (2019). fMRIPrep. *Nature Methods*. https://doi.org/10.1038/s41592-018-0235-4
- Ciric R, et al. (2017). Benchmarking confound regression strategies. *NeuroImage*. https://doi.org/10.1016/j.neuroimage.2017.03.020
- Power JD, et al. (2012). Spurious correlations from subject motion. *NeuroImage*. https://doi.org/10.1016/j.neuroimage.2011.10.018